In [ ]:
# Prathmesh Durge, 23070521109

# Credit Card Fraud Detection using Deep Learning

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, confusion_matrix

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

In [3]:
# --------------------------------------------------
# 1. Load Dataset
# --------------------------------------------------

# Download creditcard.csv and keep it in the same folder
data = pd.read_csv("creditcard.csv")

print("Dataset shape:", data.shape)
print(data.head())

Dataset shape: (284807, 31)
   Time        V1        V2        V3        V4        V5        V6        V7  \
0   0.0 -1.359807 -0.072781  2.536347  1.378155 -0.338321  0.462388  0.239599   
1   0.0  1.191857  0.266151  0.166480  0.448154  0.060018 -0.082361 -0.078803   
2   1.0 -1.358354 -1.340163  1.773209  0.379780 -0.503198  1.800499  0.791461   
3   1.0 -0.966272 -0.185226  1.792993 -0.863291 -0.010309  1.247203  0.237609   
4   2.0 -1.158233  0.877737  1.548718  0.403034 -0.407193  0.095921  0.592941   

         V8        V9  ...       V21       V22       V23       V24       V25  \
0  0.098698  0.363787  ... -0.018307  0.277838 -0.110474  0.066928  0.128539   
1  0.085102 -0.255425  ... -0.225775 -0.638672  0.101288 -0.339846  0.167170   
2  0.247676 -1.514654  ...  0.247998  0.771679  0.909412 -0.689281 -0.327642   
3  0.377436 -1.387024  ... -0.108300  0.005274 -0.190321 -1.175575  0.647376   
4 -0.270533  0.817739  ... -0.009431  0.798278 -0.137458  0.141267 -0.206010   

    

In [8]:
# --------------------------------------------------
# 2. Check Class Distribution
# --------------------------------------------------

print("\nClass Distribution:")
print(data["Class"].value_counts())

# Class:
# 0 = Normal transaction
# 1 = Fraudulent transaction


# --------------------------------------------------
# 3. Separate Features and Target
# --------------------------------------------------

X = data.drop("Class", axis=1)
y = data["Class"]


# --------------------------------------------------
# 4. Scale the Amount and Time Features
# --------------------------------------------------

scaler = StandardScaler()

X[["Time", "Amount"]] = scaler.fit_transform(
    X[["Time", "Amount"]]
)


Class Distribution:
Class
0    284315
1       492
Name: count, dtype: int64


In [9]:

# --------------------------------------------------
# 5. Split Dataset
# --------------------------------------------------

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("\nTraining samples:", X_train.shape[0])
print("Testing samples:", X_test.shape[0])

# --------------------------------------------------
# 6. Handle Imbalanced Data
# --------------------------------------------------

# Calculate class weights
classes = np.unique(y_train)

weights = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=y_train
)

class_weights = dict(zip(classes, weights))

print("\nClass Weights:")
print(class_weights)




Training samples: 227845
Testing samples: 56962

Class Weights:
{np.int64(0): np.float64(0.5008661206149896), np.int64(1): np.float64(289.14340101522845)}


In [10]:
# --------------------------------------------------
# 7. Build Deep Neural Network
# --------------------------------------------------

model = Sequential([

    Dense(
        64,
        activation="relu",
        input_shape=(X_train.shape[1],)
    ),

    Dropout(0.3),

    Dense(
        32,
        activation="relu"
    ),

    Dropout(0.3),

    Dense(
        16,
        activation="relu"
    ),

    Dense(
        1,
        activation="sigmoid"
    )
])

c:\Users\Prathmesh\AppData\Local\Programs\Python\Python310\lib\site-packages\keras\src\layers\core\dense.py:95: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [11]:

# --------------------------------------------------
# 8. Compile Model
# --------------------------------------------------

model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)


# Display model architecture
model.summary()


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 64)             │         1,984 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 16)             │           528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 1)              │            17 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 4,609 (18.00 KB)

 Trainable params: 4,609 (18.00 KB)

 Non-trainable params: 0 (0.00 B)

In [12]:

# --------------------------------------------------
# 9. Train Model
# --------------------------------------------------

early_stop = EarlyStopping(
    monitor="val_loss",
    patience=3,
    restore_best_weights=True
)

history = model.fit(
    X_train,
    y_train,
    epochs=20,
    batch_size=64,
    validation_split=0.2,
    class_weight=class_weights,
    callbacks=[early_stop],
    verbose=1
)

Epoch 1/20
2849/2849 ━━━━━━━━━━━━━━━━━━━━ 4s 1ms/step - accuracy: 0.9816 - loss: 0.3436 - val_accuracy: 0.9948 - val_loss: 0.0823
Epoch 2/20
2849/2849 ━━━━━━━━━━━━━━━━━━━━ 4s 1ms/step - accuracy: 0.9801 - loss: 0.2002 - val_accuracy: 0.9923 - val_loss: 0.0579
Epoch 3/20
2849/2849 ━━━━━━━━━━━━━━━━━━━━ 3s 1ms/step - accuracy: 0.9816 - loss: 0.1772 - val_accuracy: 0.9844 - val_loss: 0.0799
Epoch 4/20
2849/2849 ━━━━━━━━━━━━━━━━━━━━ 3s 1ms/step - accuracy: 0.9781 - loss: 0.1486 - val_accuracy: 0.9851 - val_loss: 0.0772
Epoch 5/20
2849/2849 ━━━━━━━━━━━━━━━━━━━━ 3s 1ms/step - accuracy: 0.9797 - loss: 0.1276 - val_accuracy: 0.9942 - val_loss: 0.0225
Epoch 6/20
2849/2849 ━━━━━━━━━━━━━━━━━━━━ 3s 1ms/step - accuracy: 0.9803 - loss: 0.1321 - val_accuracy: 0.9758 - val_loss: 0.1242
Epoch 7/20
2849/2849 ━━━━━━━━━━━━━━━━━━━━ 3s 1ms/step - accuracy: 0.9771 - loss: 0.1239 - val_accuracy: 0.9779 - val_loss: 0.0650
Epoch 8/20
2849/2849 ━━━━━━━━━━━━━━━━━━━━ 3s 1ms/step - accuracy: 0.9786 - loss: 0.1273 - 

In [13]:

# --------------------------------------------------
# 10. Make Predictions
# --------------------------------------------------

y_probability = model.predict(X_test)

# Convert probabilities to classes
y_pred = (y_probability >= 0.5).astype(int)


# --------------------------------------------------
# 11. Classification Report
# --------------------------------------------------

print("\nClassification Report:")

print(
    classification_report(
        y_test,
        y_pred,
        target_names=["Normal", "Fraud"]
    )
)



1781/1781 ━━━━━━━━━━━━━━━━━━━━ 1s 492us/step

Classification Report:
              precision    recall  f1-score   support

      Normal       1.00      0.99      1.00     56864
       Fraud       0.22      0.90      0.35        98

    accuracy                           0.99     56962
   macro avg       0.61      0.95      0.67     56962
weighted avg       1.00      0.99      1.00     56962



In [14]:

# --------------------------------------------------
# 12. Confusion Matrix
# --------------------------------------------------

print("\nConfusion Matrix:")

cm = confusion_matrix(y_test, y_pred)

print(cm)


# --------------------------------------------------
# 13. Extract Precision, Recall and F1
# --------------------------------------------------

report = classification_report(
    y_test,
    y_pred,
    output_dict=True
)

print("\nFraud Detection Metrics:")

print("Precision:",
      round(report["1"]["precision"], 4))

print("Recall:",
      round(report["1"]["recall"], 4))

print("F1-Score:",
      round(report["1"]["f1-score"], 4))


Confusion Matrix:
[[56544   320]
 [   10    88]]

Fraud Detection Metrics:
Precision: 0.2157
Recall: 0.898
F1-Score: 0.3478
